In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run /Workspace/Users/dhotepatil00@gmail.com/regis-healthcare/1_setup/utility

In [0]:
print(bronze_schema,silver_schema,gold_schema) 

In [0]:
dbutils.widgets.text("catalog","regis_healthcare","catalog")
dbutils.widgets.text("data_source","incidents","data_source")

In [0]:
catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

#### Silver Processing

In [0]:
df_bronze = spark.sql(f"select * from {catalog}.{bronze_schema}.{data_source};")
display(df_bronze)
print(df_bronze.count())

In [0]:
# schema check
print(df_bronze.count())
df_bronze.printSchema()

In [0]:
df_bronze.columns

In [0]:
# drop duplicate
df_silver = df_bronze.dropDuplicates()
print(df_silver.count())

In [0]:
# incident_id
df_silver = df_silver.withColumn(
    "incident_id",
    F.trim(F.col("incident_id"))
#  resident_id
).withColumn(
    "resident_id",
    F.trim(F.col("resident_id"))
# facility_id
).withColumn(
    "facility_id",
    F.trim(F.col("facility_id"))
#  employee_id
).withColumn(
    "employee_id",
    F.trim(F.col("employee_id"))
#  incident_date
).withColumn(
    "incident_date",
    F.trim(F.col("incident_date"))
#  incident_type
).withColumn(
    "incident_type",
    F.trim(F.col("incident_type"))
#  severity
).withColumn(
    "severity",
    F.trim(F.col("severity"))
#  description
).withColumn(
    "description",
    F.trim(F.col("description"))
#  reported_by
).withColumn(
    "reported_by",
    F.trim(F.col("reported_by"))
#  action_taken
).withColumn(
    "action_taken",
    F.trim(F.col("action_taken"))
#  follow_up_required
).withColumn(
    "follow_up_required",
    F.trim(F.col("follow_up_required"))
#  created_at
).withColumn(
    "created_at",
    F.trim(F.col("created_at"))
)

In [0]:
# null records count 
from pyspark.sql.functions import col,count,when
null_count = df_silver.select([count(when(col(c).isNull(),c)).alias(c)for c in df_silver.columns
                               ])
display(null_count)

#### Cleaning data in table

In [0]:
# incident_id

from pyspark.sql.functions import col,when
df_filt = df_silver.filter(~col("incident_id").rlike("^INC"))

df_silver = df_silver.withColumn(
    "incident_id",
    when(
        (col("incident_id").isNull()) | (~col("incident_id").rlike("^INC")),
        "0"
    ).otherwise(col("incident_id"))
)

display(df_filt)
display(df_silver)

In [0]:
#  resident_id

from pyspark.sql.functions import col,when
df_filt = df_silver.filter(~col("resident_id").rlike("^RES"))

df_silver = df_silver.withColumn(
    "resident_id",
    when(
        (col("resident_id").isNull()) | (~col("resident_id").rlike("^RES")),
        "0"
    ).otherwise(col("resident_id"))
)

display(df_filt)
display(df_silver)

In [0]:
# facility_id
from pyspark.sql.functions import col,when
df_filt = df_silver.filter(~col("facility_id").rlike("^FAC"))

df_silver = df_silver.withColumn(
    "facility_id",
    when(
        (col("facility_id").isNull()) | (~col("facility_id").rlike("^FAC")),
        "0"
    ).otherwise(col("facility_id"))
)

display(df_filt)
display(df_silver)

In [0]:
#  employee_id
from pyspark.sql.functions import col,when
df_filt = df_silver.filter(~col("employee_id").rlike("^EMP"))

df_silver = df_silver.withColumn(
    "employee_id",
    when(
        (col("employee_id").isNull()) | (~col("employee_id").rlike("^EMP")),
        "0"
    ).otherwise(col("employee_id"))
)

display(df_filt)
display(df_silver)

In [0]:
#  incident_date

from pyspark.sql.functions import to_timestamp
df_silver = df_silver.withColumn("incident_date",when(~col("incident_date").rlike("[-_=\\[\\(<\\>\\?#*~%$&@]"),None).otherwise(col("incident_date")))

df_invalid = df_silver.filter(~col("incident_date").rlike("[-_=\\[\\(<\\>\\?#*~%$&@]"))
display(df_invalid)
df_invalid = df_silver.groupBy("incident_date").count().filter(col("count")>1)
display(df_invalid)

filt_xx = {'9999-99-99':None,
           'not-a-date' : None,
           '2030-01-01': None,
           '2026-02-30':None}
df_silver = df_silver.replace(filt_xx,subset = ["incident_date"])

df_silver = df_silver.withColumn("incident_date",to_timestamp(col("incident_date"),"yyyy-MM-dd HH:mm:ss"))
display(df_silver)

In [0]:
#  incident_type

from pyspark.sql.functions import col,when,upper,trim
df_silver = df_silver.withColumn("incident_type",upper(trim(col("incident_type"))))
dup = df_silver.groupBy("incident_type").count().filter(col("count")>1)
# display(dup)
replexx = {
"NAN" : "UNKNOWN",
"#N/A" : "UNKNOWN",
"NONE" : "UNKNOWN",
"NULL" : "UNKNOWN",
"N/A" : "UNKNOWN",
"UNKNOWN" : "UNKNOWN",
"" : "UNKNOWN"
}
df_silver = df_silver.replace(replexx,subset = ["incident_type"])
df_silver = df_silver.fillna({"incident_type":"UNKNOWN"})

dup = df_silver.groupBy("incident_type").count().filter(col("count")>1)
display(dup)

In [0]:
#  severity
from pyspark.sql.functions import col,when,initcap,trim
df_silver = df_silver.withColumn("severity",initcap(trim(col("severity"))))
dup = df_silver.groupBy("severity").count().filter(col("count")>1)
display(dup)

In [0]:
#  description

from pyspark.sql.functions import col,when,initcap,trim
df_silver = df_silver.withColumn("description",initcap(trim(col("description"))))
dup = df_silver.groupBy("description").count().filter(col("count")>1)
# display(dup)

replexx_Z = {"Null" : "Unknown",
    "#n/a" : "Unknown",
"Unknown" : "Unknown",
"Nan" : "Unknown",
"N/a" : "Unknown",
"null" : "Unknown",
"None" : "Unknown",
"" : "Unknown"}
df_silver = df_silver.replace(replexx_Z,subset=["description"])
df_silver = df_silver.fillna({"description":"Unknown"})
dup = df_silver.groupBy("description").count().filter(col("count")>1)
display(dup)


In [0]:
#  reported_by

df_invalid = df_silver.filter(col("reported_by").rlike("[-_=\\[\\(<\\>\\?#*~%$&@]"))
display(df_invalid)
from pyspark.sql.functions import col,when,initcap,trim
df_silver = df_silver.withColumn("reported_by",initcap(trim(col("reported_by"))))
dup = df_silver.groupBy("reported_by").count().filter(col("count")>1)
display(dup)

In [0]:
#  action_taken

from pyspark.sql.functions import col,when,initcap,trim
df_silver = df_silver.withColumn("action_taken",initcap(trim(col("action_taken"))))

repl_xcvz = {"Null" : "Unknown",
"Nan" : "Unknown",
"null" : "Unknown",
"N/a" : "Unknown",
"None" : "Unknown",
"Unknown" : "Unknown",
"#n/a" : "Unknown",
"": "Unknown"
}
df_silver = df_silver.replace(repl_xcvz,subset = ["action_taken"])
df_silver = df_silver.fillna({"action_taken":"Unknown"})

df_invalid = df_silver.filter(col("action_taken").rlike("[-_=\\[\\(<\\>\\?#*~%$&@]"))
display(df_invalid)

dup = df_silver.groupBy("action_taken").count().filter(col("count")>1)
display(dup)


In [0]:
#  follow_up_required

from pyspark.sql.functions import col,when,initcap,trim
df_silver = df_silver.withColumn("follow_up_required",when(col("follow_up_required")=="Yes","Yes").when(col("follow_up_required")=="No","No").otherwise("Unknown"))
dup = df_silver.groupBy("follow_up_required").count().filter(col("count")>1)
display(dup)


In [0]:
#  created_at
dup= df_silver.groupBy("created_at").count()
# display(dup)

df_silver = df_silver.withColumn("created_at",to_timestamp(col("created_at"),"yyyy-MM-dd HH:mm:ss"))
display(df_silver)

#### Silver table load

In [0]:
df_silver.write\
    .format("delta")\
        .option("delta.enableChangeDataFeed","true")\
            .option("mergeSchema","true")\
                .option("overwriteSchema","true")\
            .mode("overwrite")\
               .saveAsTable(f"{catalog}.{silver_schema}.{data_source}")

dt = spark.sql(f"select * from {catalog}.{silver_schema}.{data_source};")
print(dt.count())
display(dt)

In [0]:
# load to s3
df_silver.write.format("delta")\
    .option("mergeSchema","true")\
    .option("overwriteSchema","true")\
    .mode("overwrite")\
    .partitionBy("current_date")\
    .save(f"s3://regis-healthcare/silver-clean-data/{data_source}/")